# `aidp-rest-generic` live test — `aidataplatform` `type=GENERIC_REST`
Reads from any REST endpoint that publishes a `manifest.url`.


In [ ]:
import sys, os
# Adjust this if you've uploaded the plugin scripts/ dir elsewhere.
sys.path.insert(0, '/Workspace/Shared/oracle_ai_data_platform_connectors/scripts')


In [ ]:
from oracle_ai_data_platform_connectors.aidataplatform import (
    AIDP_FORMAT, aidataplatform_options,
)
extra = {
    'base.url':     os.environ['REST_BASE_URL'],
    'manifest.url': os.environ['REST_MANIFEST_URL'],
    'auth.type':    'basic',
    'api':          os.environ['REST_API'],
}
# Each REST_PROP_* env var becomes a derived.property.* option.
for k, v in os.environ.items():
    if k.startswith('REST_PROP_'):
        extra[f'derived.property.{k[len("REST_PROP_"):]}'] = v
opts = aidataplatform_options(
    type='GENERIC_REST',
    user=os.environ['REST_USER'],
    password=os.environ['REST_PASSWORD'],
    schema=os.environ.get('REST_SCHEMA', 'default'),
    extra=extra,
)


In [ ]:
df = spark.read.format(AIDP_FORMAT).options(**opts).load()
df.show(5)


In [ ]:
# Live-test driver parses this final cell's stdout for the JSON summary.
import json, time
summary = {
    'connector': 'aidp-rest-generic',
    'auth': 'basic',
    'rows': int(df.count()),
    'schema': sorted([f.name for f in df.schema.fields]),
    'timestamp_utc': int(time.time()),
}
print('AIDP_LIVE_TEST_RESULT_BEGIN')
print(json.dumps(summary, indent=2))
print('AIDP_LIVE_TEST_RESULT_END')
